In [ ]:
import os, time
import pandas as pd
from dotenv import load_dotenv
from sodapy import Socrata

load_dotenv()
client = Socrata("data.cityofnewyork.us", os.getenv("SOCRATA_TOKEN"), timeout=120)

os.makedirs("ems_raw", exist_ok=True)

cols = "incident_datetime,zipcode,initial_severity_level_code,valid_incident_rspns_time_indc"
PAGE = 10000   # smaller pages are far less likely to time out into a 503

def get_page(where, offset, retries=8):
    for i in range(retries):
        try:
            return client.get("76xm-jjuj", select=cols, where=where,
                              order="incident_datetime", limit=PAGE, offset=offset)
        except Exception as e:
            wait = min(2 ** i, 60)
            print(f"  offset {offset}: {e} -> retry {i+1}/{retries} in {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"Gave up at offset {offset}")

for year in range(2016, 2023):
    final = f"ems_raw/{year}.csv"
    if os.path.exists(final):
        continue
    part_dir = f"ems_raw/{year}_parts"
    os.makedirs(part_dir, exist_ok=True)
    w = f"incident_datetime between '{year}-01-01T00:00:00' and '{year}-12-31T23:59:59'"

    offset = 0
    while True:
        part = f"{part_dir}/{offset:09d}.csv"
        if os.path.exists(part):                  # page already downloaded
            offset += PAGE
            continue
        batch = get_page(w, offset)
        if not batch:
            break
        pd.DataFrame.from_records(batch).to_csv(part, index=False)
        offset += PAGE
        print(f"{year}: {offset:,} rows requested")

    parts = sorted(os.listdir(part_dir))
    if parts:
        pd.concat((pd.read_csv(f"{part_dir}/{p}", dtype={"zipcode": str}) for p in parts),
                  ignore_index=True).to_csv(final, index=False)
        print(year, "saved")

2017: 10,000 rows requested
2017: 20,000 rows requested
2017: 30,000 rows requested
2017: 40,000 rows requested
2017: 50,000 rows requested
2017: 60,000 rows requested
2017: 70,000 rows requested
2017: 80,000 rows requested
2017: 90,000 rows requested
2017: 100,000 rows requested
2017: 110,000 rows requested
2017: 120,000 rows requested
2017: 130,000 rows requested
2017: 140,000 rows requested
2017: 150,000 rows requested
2017: 160,000 rows requested
2017: 170,000 rows requested
2017: 180,000 rows requested
2017: 190,000 rows requested
2017: 200,000 rows requested
2017: 210,000 rows requested
2017: 220,000 rows requested
2017: 230,000 rows requested
2017: 240,000 rows requested
2017: 250,000 rows requested
2017: 260,000 rows requested
2017: 270,000 rows requested
2017: 280,000 rows requested
2017: 290,000 rows requested
2017: 300,000 rows requested
2017: 310,000 rows requested
2017: 320,000 rows requested
2017: 330,000 rows requested
2017: 340,000 rows requested
2017: 350,000 rows requ

In [ ]:
import pandas as pd

df = pd.concat(
    (pd.read_csv(f"ems_raw/{y}.csv", dtype={"zipcode": str}) for y in range(2016, 2023)),
    ignore_index=True,
)
df["incident_datetime"] = pd.to_datetime(df["incident_datetime"], errors="coerce")

print(df.shape)
print(df["incident_datetime"].dt.year.value_counts().sort_index())

In [ ]:

df = df.drop(columns=["incident_id", "reopen_indicator", "special_event_indicator", "standby_indicator", "transfer_indicator", "congressionaldistrict", "communityschooldistrict", "citycouncildistrict", "policeprecinct", "incident_disposition_code", "incident_dispatch_area", "communitydistrict"], errors="ignore")   # these columns are not in the 4-column download

In [ ]:
dt_cols = [c for c in ["incident_datetime", "first_assignment_datetime", "first_on_scene_datetime"] if c in df.columns]
for c in dt_cols:
    df[c] = pd.to_datetime(df[c], errors="coerce")

num_cols = [c for c in ["incident_response_seconds_qy", "incident_travel_tm_seconds_qy"] if c in df.columns]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

In [ ]:
df.head()

In [ ]:
import pandas as pd, numpy as np

df["incident_datetime"] = pd.to_datetime(df["incident_datetime"], errors="coerce")
df = df.dropna(subset=["incident_datetime", "zipcode"])
df["bucket"] = df["incident_datetime"].dt.floor("30min")

counts = df.groupby(["zipcode", "bucket"]).size().rename("y")

# Full grid so quiet periods appear as zeros
zones = df["zipcode"].unique()
buckets = pd.date_range(df["bucket"].min(), df["bucket"].max(), freq="30min")
idx = pd.MultiIndex.from_product([zones, buckets], names=["zipcode", "bucket"])
panel = counts.reindex(idx, fill_value=0).reset_index()
panel = panel.sort_values(["zipcode", "bucket"]).reset_index(drop=True)

In [ ]:
g = panel.groupby("zipcode")["y"]

for lag in [1, 2, 3, 6, 48, 336]:        # 30 min, 1h, 1.5h, 3h, 1 day, 1 week
    panel[f"lag_{lag}"] = g.shift(lag)

panel["roll_2h"]  = g.transform(lambda s: s.shift(1).rolling(4).sum())
panel["roll_6h"]  = g.transform(lambda s: s.shift(1).rolling(12).sum())
panel["roll_24h"] = g.transform(lambda s: s.shift(1).rolling(48).sum())

t = panel["bucket"]
panel["hour"] = t.dt.hour + t.dt.minute / 60
panel["dow"] = t.dt.dayofweek
panel["month"] = t.dt.month
panel["is_weekend"] = (panel["dow"] >= 5).astype(int)
panel["zipcode"] = panel["zipcode"].astype("category")

## Link all other data sources into one big table (`full`)
EMS counts (`panel`) are combined with everything fetched and cleaned by **`DarraghC.ipynb`** (run that notebook first; its outputs live in `notebooks/data/`):

| Joined on | From | Columns added |
|---|---|---|
| 30-minute `bucket` | `weather_calendar_30min.csv.gz` | temperature, feels-like, humidity, rain, wind, cloud, hot / cold / heat-stress flags, holidays, and the `eval_` heat / heavy-rain day flags |
| `zipcode` -> `zone` -> | `interim/zip_to_zone.csv` then `interim/zone_context.csv` | `zone`, `borough`, `lon`, `lat`, `pop_est`, `asthma_rate_per_10k`, Census columns (when available), hospitals / firehouses in the zone, distance to nearest hospital |

**`panel` and the model cells below are not modified.** The model builds its feature list from every column of `panel`, so merging into `panel` would silently change the model
(and string columns would break LightGBM). Everything lives in the separate table `full` until we decide which columns to use.
`eval_*` columns slice results by heat or heavy-rain days and must never be model inputs. The hospital / firehouse list (`interim/posts.csv`) and the FDNY monthly benchmark
(`interim/fdny_monthly.csv`) are reference tables for the simulator, not row-level joins.

In [ ]:
from pathlib import Path

DATA = Path("data")                                   # notebooks/data, written by DarraghC.ipynb
needed = {"weather": DATA / "weather_calendar_30min.csv.gz",
          "zip_to_zone": DATA / "interim" / "zip_to_zone.csv",
          "zone_context": DATA / "interim" / "zone_context.csv"}
missing = [str(f) for f in needed.values() if not f.exists()]
assert not missing, f"Run DarraghC.ipynb first, it creates: {missing}"

wx = pd.read_csv(needed["weather"], parse_dates=["bucket"]).set_index("bucket")
zip_to_zone = pd.read_csv(needed["zip_to_zone"], dtype=str).set_index("zipcode")["zone"]
zone_ctx = pd.read_csv(needed["zone_context"], dtype={"zone": str}).set_index("zone").drop(columns=["label"])

full = panel.copy()                                   # same rows and columns as panel, plus the new columns below
for c in full.select_dtypes("float64").columns:       # halve memory: the panel is tens of millions of rows
    full[c] = full[c].astype("float32")

# 1) EMS zipcode -> zone (look up once per ZIP, not once per row)
zips = full["zipcode"].cat.categories
zone_of_zip = pd.Series(zips, index=zips).map(zip_to_zone)
zone_names = pd.Index(sorted(zone_of_zip.dropna().unique()))
zip_to_code = np.array([zone_names.get_loc(z) if pd.notna(z) else -1 for z in zone_of_zip], dtype="int16")
zone_code = zip_to_code[full["zipcode"].cat.codes.to_numpy()]            # -1 = ZIP is not in any zone
full["zone"] = pd.Categorical.from_codes(zone_code, categories=zone_names)

# 2) zone context (borough, health context, access to care, ...)
for c in zone_ctx.columns:
    vals = zone_ctx[c].reindex(zone_names)
    if pd.api.types.is_numeric_dtype(vals):
        arr = vals.to_numpy(dtype="float32")
        full[c] = np.where(zone_code >= 0, arr[zone_code], np.float32("nan")).astype("float32")
    else:                                             # text column such as borough -> category
        cats = pd.Index(sorted(vals.dropna().unique()))
        lut = np.array([cats.get_loc(v) if pd.notna(v) else -1 for v in vals], dtype="int16")
        full[c] = pd.Categorical.from_codes(np.where(zone_code >= 0, lut[zone_code], -1), categories=cats)

# 3) weather + calendar on the 30-minute bucket
pos = wx.index.get_indexer(full["bucket"])            # -1 where a bucket has no weather row
for c in wx.columns:
    arr = wx[c].to_numpy(dtype="float32")
    full[c] = np.where(pos >= 0, arr[pos], np.float32("nan")).astype("float32")

EVAL_COLS = [c for c in full.columns if c.startswith("eval_")]
print("full:", full.shape, f"| ~{full.memory_usage().sum() / 1e9:.1f} GB in memory | panel is unchanged: {panel.shape}")
print(f"rows whose ZIP is in no zone: {(zone_code < 0).mean():.2%} | rows with no weather: {(pos < 0).mean():.2%}")
print("slice results by these, never use as model inputs:", EVAL_COLS)
print("added columns:", [c for c in full.columns if c not in panel.columns])
full.head(3).T

In [73]:
train_end = "2022-12-31"
train = panel[panel["bucket"] <= train_end]
test = panel[panel["bucket"] > train_end]

hist = (train.groupby(["zipcode", "dow", "hour"], observed=True)["y"]
              .mean().rename("baseline").reset_index())
test = test.merge(hist, on=["zipcode", "dow", "hour"], how="left")

In [74]:
import lightgbm as lgb

features = [c for c in panel.columns if c not in ["y", "bucket"]]
train_df = train.dropna(subset=["lag_336"])      # drop rows without full history
start, end = panel["bucket"].min(), panel["bucket"].max()
span = end - start
train_end = start + span * 0.75
val_cut   = start + span * 0.60

train = panel[panel["bucket"] <= train_end]
test  = panel[panel["bucket"] >  train_end]

train_df = train.dropna(subset=["lag_336"])
tr = train_df[train_df["bucket"] <  val_cut]
va = train_df[train_df["bucket"] >= val_cut]
print(tr.shape, va.shape, test.shape)   # all three should be non-empty

model = lgb.LGBMRegressor(
    objective="poisson",
    n_estimators=2000, learning_rate=0.03,
    num_leaves=63, min_child_samples=50,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
)
model.fit(
    tr[features], tr["y"],
    eval_X=va[features], 
    eval_y=va["y"],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)],
)
test["pred"] = model.predict(test[features])

(132000, 16) (51480, 16) (85800, 16)
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008443 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 444
[LightGBM] [Info] Number of data points in the train set: 132000, number of used features: 13
[LightGBM] [Info] Start training from score -1.320138
Training until validation scores don't improve for 100 rounds
[200]	valid_0's poisson: 0.49269
[400]	valid_0's poisson: 0.490427
Early stopping, best iteration is:
[376]	valid_0's poisson: 0.490378


In [75]:
print(panel["bucket"].min(), panel["bucket"].max())
print(panel.shape)
print(panel.isna().sum().sort_values(ascending=False).head(8))
print(train.shape, train_df.shape, tr.shape, va.shape)

2005-01-01 00:00:00 2005-02-02 11:30:00
(343200, 16)
lag_336     73920
lag_48      10560
roll_24h    10560
roll_6h      2640
lag_6        1320
roll_2h       880
lag_3         660
lag_2         440
dtype: int64
(257400, 16) (183480, 16) (132000, 16) (51480, 16)


In [76]:
hist = (train.groupby(["zipcode", "dow", "hour"], observed=True)["y"]
              .mean().rename("baseline").reset_index())

test = test.merge(hist, on=["zipcode", "dow", "hour"], how="left")

In [77]:
from sklearn.metrics import mean_poisson_deviance, mean_absolute_error

t2 = test.dropna(subset=["pred", "baseline"])
print("MAE  model:", mean_absolute_error(t2.y, t2.pred),
      " baseline:", mean_absolute_error(t2.y, t2.baseline))
print("Poisson deviance model:", mean_poisson_deviance(t2.y, t2.pred.clip(1e-6)),
      " baseline:", mean_poisson_deviance(t2.y, t2.baseline.clip(1e-6)))

MAE  model: 0.34061848333392075  baseline: 0.3557886557886558
Poisson deviance model: 0.6437258992803876  baseline: 2.242235323119212


In [78]:
print(features)

['zipcode', 'lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_48', 'lag_336', 'roll_2h', 'roll_6h', 'roll_24h', 'hour', 'dow', 'month', 'is_weekend']


In [80]:
print(df)

        incident_datetime initial_call_type  initial_severity_level_code  \
0     2005-01-01 00:00:24            INJURY                            5   
1     2005-01-01 00:00:25             OTHER                            6   
2     2005-01-01 00:00:43            RESPIR                            4   
3     2005-01-01 00:00:37             OBMIS                            4   
4     2005-01-01 00:00:58            INJMIN                            7   
...                   ...               ...                          ...   
99995 2005-02-02 11:35:06              SICK                            6   
99996 2005-02-02 11:35:12            MVAINJ                            4   
99997 2005-02-02 11:35:30            INJURY                            4   
99998 2005-02-02 11:36:38             ABDPN                            5   
99999 2005-02-02 11:36:41              SICK                            4   

      final_call_type  final_severity_level_code first_assignment_datetime  \
0        